# ✅ 회귀(Regression) 문제 템플릿

## 이 파일을 쓰는 상황
```
정답(Target)이 연속된 숫자 → 예) 423초, 15만원, 자전거 37대
평가지표가 RMSE, MAE, RMSLE
```

## ★ 표시된 곳만 실제 문제에 맞게 바꾸면 됩니다!

## 전체 흐름
```
[Step1] 도구 꺼내기 (라이브러리 불러오기)
[Step2] 데이터 보기 (EDA)
[Step3] 데이터 손질 (전처리)
[Step4] AI 모델 학습 + 평가
[Step5] 테스트 데이터 예측
[Step6] 제출 파일 만들기
```

---
## [Step1] 도구 꺼내기

```
요리하기 전에 냄비, 칼, 도마를 꺼내는 것처럼
코딩하기 전에 필요한 도구(라이브러리)를 먼저 불러옵니다.
```

In [ ]:
# 기본 도구 (데이터 다루기)
import numpy as np
import pandas as pd

# 그래프 그리기
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 끄기

In [ ]:
# ★ 회귀 모델들 (이 중에서 가장 좋은 것을 골라 씁니다)
from sklearn.linear_model import LinearRegression        # 선형 회귀
from sklearn.tree import DecisionTreeRegressor           # 결정 트리
from sklearn.ensemble import RandomForestRegressor       # 랜덤 포레스트
from sklearn.ensemble import ExtraTreesRegressor         # 엑스트라 트리
from xgboost import XGBRegressor                         # XGBoost
from lightgbm import LGBMRegressor                       # LightGBM

# 데이터 나누기 / 평가 / 튜닝 도구
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import MinMaxScaler

### 데이터 불러오기

In [ ]:
train = pd.read_csv('./data/train.csv')  # 정답이 있는 학습용 데이터
test  = pd.read_csv('./data/test.csv')   # 정답 없는 예측용 데이터

print('train 크기:', train.shape)
print('test  크기:', test.shape)
train.head()

In [ ]:
train.info()

In [ ]:
test.info()

### 날짜형 변수가 있을 때만! — 월/일/시간/요일로 분리

```
컴퓨터는 '2016-01-18 11:10'이라는 날짜를 이해 못 합니다.
그래서 월(1월), 일(18일), 시간(11시), 요일(월요일=0) 등으로 쪼갭니다.
날짜 컬럼이 없으면 이 셀은 무시하세요!
```

In [ ]:
# ★ 날짜형 컬럼이 있는 경우만 주석 해제해서 사용
# DATE_COL = 'pickup_datetime'  # ★ 날짜 컬럼명 변경

# for df in [train, test]:
#     df[DATE_COL] = pd.to_datetime(df[DATE_COL])   # 날짜형으로 변환
#     df['month'] = df[DATE_COL].dt.month            # 월 (1~12)
#     df['day']   = df[DATE_COL].dt.day              # 일 (1~31)
#     df['hour']  = df[DATE_COL].dt.hour             # 시간 (0~23)
#     df['dow']   = df[DATE_COL].dt.dayofweek        # 요일 (월=0, 일=6)
#     df.drop([DATE_COL], axis=1, inplace=True)      # 원래 날짜 컬럼 삭제

print('날짜 변수 없으면 이 셀 무시!')

### 컬럼 처리 방향 자동 판별

```
이 코드를 실행하면 각 컬럼을 어떻게 처리할지 자동으로 알려줍니다!

❌ 삭제    → drop_cols에 넣기
🔤 OHE    → ohe_cols에 넣기 (글자 → 숫자 번역)
📊 스케일링 → scale_cols에 넣기 (0~1 범위로)
📅 날짜 분리 → 위 셀에서 분리 완료하면 month/day/hour/dow가 🔤로 나옴
⚠️ 결측   → 채워줘야 함
```

In [ ]:
def analyze_columns(df, target_col, drop_candidates=None):
    if drop_candidates is None:
        drop_candidates = []
    print(f"{'컬럼명':22} {'타입':10} {'고유값수':>8} {'결측수':>6}  처리 방향")
    print('-' * 80)
    for col in df.columns:
        if col == target_col:
            print(f"{col:22} {'':10} {'':>8} {'':>6}  🎯 TARGET (정답 — 건드리지 않음)")
            continue
        dtype    = str(df[col].dtype)
        n_unique = df[col].nunique()
        n_null   = df[col].isnull().sum()
        if col in drop_candidates:
            direction = '❌ 삭제'
        elif dtype == 'object':
            direction = '🔤 OHE (글자형 → 숫자로 번역)' if n_unique <= 20 else '❌ 삭제 검토 (고유값 너무 많음)'
        elif 'datetime' in dtype:
            direction = '📅 날짜 분리 후 OHE (월/일/시/요일 추출)'
        elif n_unique <= 10:
            direction = f'🔤 OHE 검토 (숫자지만 {n_unique}가지뿐 → 카테고리 가능)'
        else:
            direction = '📊 MinMaxScaler (연속 숫자 → 0~1로 맞추기)'
        null_mark = f'  ⚠️ 결측 {n_null}개 — 채워줘야 함!' if n_null > 0 else ''
        print(f"{col:22} {dtype:10} {n_unique:>8} {n_null:>6}  {direction}{null_mark}")

# ★ target_col과 drop_candidates를 실제 문제에 맞게 바꾸고 실행!
TARGET = 'trip_duration'    # ★ 정답 컬럼명 변경
analyze_columns(train, target_col=TARGET,
                drop_candidates=['id', 'dropoff_datetime'])  # ★ 삭제 대상 변경

---
## [Step2] 데이터 보기 (EDA)

```
요리 전에 재료를 살펴보는 단계입니다.
특히 회귀는 Target(정답)의 분포가 중요합니다.
값이 한쪽으로 너무 치우쳐 있으면 로그 변환이 필요할 수 있습니다.
```

In [ ]:
train.describe()

### Target 분포 확인 — 로그 변환이 필요한지 보기

```
왼쪽(원본)이 한쪽으로 치우쳐 있고
오른쪽(로그변환)이 더 종 모양에 가까우면 → 로그 변환 하세요!

특히 평가지표가 RMSLE이면 반드시 로그 변환!
```

In [ ]:
TARGET = 'trip_duration'  # ★ 정답 컬럼명 변경

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET],           ax=axes[0]).set_title('원본 분포')
sns.histplot(np.log1p(train[TARGET]), ax=axes[1]).set_title('log1p 변환 후 (더 종 모양이면 변환 사용)')
plt.tight_layout()
plt.show()

### 양적변수 분포 확인

In [ ]:
# ★ 📊 표시된 컬럼들로 변경
num_cols = ['passenger_count', 'pickup_longitude', 'pickup_latitude']

fig, axes = plt.subplots(1, len(num_cols), figsize=(5*len(num_cols), 4))
for i, col in enumerate(num_cols):
    sns.histplot(train[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 질적변수 분포 확인

In [ ]:
# ★ 🔤 표시된 컬럼들로 변경
cat_cols = ['vendor_id', 'store_and_fwd_flag']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5*len(cat_cols), 4))
for i, col in enumerate(cat_cols):
    sns.countplot(x=col, data=train, ax=axes[i])
plt.tight_layout()
plt.show()

### 추가 변수 생성 — 필요할 때만!

```
위도/경도 좌표가 있으면 '이동 거리'를 직접 계산해서 새 컬럼으로 만들 수 있습니다.
이런 식으로 기존 컬럼을 조합해 새로운 의미 있는 값을 만드는 것을 '파생변수'라고 합니다.
필요 없으면 이 셀 무시!
```

In [ ]:
# 예시: 승차/하차 위경도 → 직선 거리 계산
# for df in [train, test]:
#     dist_lon = df['pickup_longitude'] - df['dropoff_longitude']
#     dist_lat = df['pickup_latitude']  - df['dropoff_latitude']
#     df['distance'] = np.sqrt(dist_lon**2 + dist_lat**2)

print('파생변수 불필요하면 이 셀 무시!')

---
## [Step3] 데이터 손질 (전처리)

### (1) 중복값 제거 — train만!

In [ ]:
print('중복 행 수:', train.duplicated().sum())
train.drop_duplicates(inplace=True)

### (2) 불필요한 컬럼 삭제

```
analyze_columns()에서 ❌ 표시된 것들을 drop_cols에 넣으세요.
```

In [ ]:
# ★ ❌ 표시된 컬럼들 입력
drop_cols = ['id']  # ★ 변경

train.drop(drop_cols, axis=1, inplace=True)
test.drop(drop_cols,  axis=1, inplace=True)
print('삭제 후 크기:', train.shape, test.shape)

### (3) 결측치(빈칸) 채우기

```
📊 양적변수(숫자) → 평균값으로 채우기
🔤 질적변수(글자) → 가장 많이 나온 값으로 채우기
⚠️ 항상 train으로 기준값 계산! test는 train 기준값을 그대로 적용!
```

In [ ]:
print('=== train 결측치 ==='); print(train.isnull().sum())
print('=== test  결측치 ==='); print(test.isnull().sum())

In [ ]:
# ★ 결측치 있는 컬럼에 맞게 수정

# 📊 양적변수 결측 → 평균으로 채우기
# col_mean = train['컬럼명'].mean()
# train['컬럼명'].fillna(col_mean, inplace=True)
# test['컬럼명'].fillna(col_mean, inplace=True)

# 🔤 질적변수 결측 → 최빈값으로 채우기
# col_mode = train['컬럼명'].mode()[0]
# train['컬럼명'].fillna(col_mode, inplace=True)
# test['컬럼명'].fillna(col_mode, inplace=True)

print('결측치 없으면 이 셀 무시!')

### (4) 아웃라이어(이상치) 제거 — train & 양적변수만!

```
회귀 문제는 Target(정답)도 아웃라이어 제거 대상입니다!
(분류와 다른 점!)

왜? 정답이 '432초'인데 딱 하나가 '86000초'이면
    AI가 "가끔 86000초도 나올 수 있구나"라고 잘못 배웁니다.

IQR 방법 = 자동으로 이상한 범위 계산해서 제거
```

In [ ]:
TARGET = 'trip_duration'  # ★ 정답 컬럼명 변경

# boxplot으로 아웃라이어 확인 (박스 밖으로 튀는 점 = 아웃라이어)
plt.figure(figsize=(5, 3))
sns.boxplot(y=TARGET, data=train)
plt.title('아웃라이어 확인 — 점이 많이 보이면 제거 필요')
plt.show()

In [ ]:
# IQR 방법으로 아웃라이어 제거 (★ Target 컬럼명 변경)
TARGET = 'trip_duration'

Q1  = train[TARGET].quantile(0.25)  # 하위 25% 값
Q3  = train[TARGET].quantile(0.75)  # 상위 25% 값
IQR = Q3 - Q1                       # 범위
MIN = Q1 - 1.5 * IQR                # 이 값보다 작으면 이상치
MAX = Q3 + 1.5 * IQR                # 이 값보다 크면 이상치

train = train[(train[TARGET] >= MIN) & (train[TARGET] <= MAX)]
print(f'아웃라이어 제거 후 train: {train.shape}  (남은 데이터 수)')

In [ ]:
# Feature 컬럼에도 필요하면 동일하게 적용
# col = '컬럼명'  # ★ 변경
# Q1, Q3 = train[col].quantile(0.25), train[col].quantile(0.75)
# IQR = Q3 - Q1
# train = train[(train[col] >= Q1-1.5*IQR) & (train[col] <= Q3+1.5*IQR)]

### (5) Feature(입력) / Target(정답) 분리

In [ ]:
TARGET = 'trip_duration'  # ★ 정답 컬럼명 변경

Xtrain = train.drop([TARGET], axis=1)
ytrain  = train[TARGET]
Xtest  = test.copy()

print('Xtrain:', Xtrain.shape, '  ytrain:', ytrain.shape, '  Xtest:', Xtest.shape)

### (6) OHE — 글자를 숫자로 번역

```
analyze_columns()에서 🔤 표시된 컬럼들을 ohe_cols에 넣으세요.
```

In [ ]:
# ★ 🔤 표시된 컬럼들 입력 (Target 제외!)
ohe_cols = ['vendor_id', 'store_and_fwd_flag']  # ★ 변경

Xtrain_ohe = pd.get_dummies(Xtrain, columns=ohe_cols)
Xtest_ohe  = pd.get_dummies(Xtest,  columns=ohe_cols)

print('OHE 후 컬럼 수 - Xtrain:', Xtrain_ohe.shape[1], '  Xtest:', Xtest_ohe.shape[1])

### (7) 스케일링 — 숫자 단위 통일

```
⚠️ fit_transform(train) 기준 계산 + 변환
   transform(test)       변환만! (계산 다시 하면 안 됨)
```

In [ ]:
minmax_scaler = MinMaxScaler()

# OHE 후 전체 컬럼 스케일링
Xtrain_ohe_scaled = pd.DataFrame(
    data=minmax_scaler.fit_transform(Xtrain_ohe),  # train: 기준 계산 + 변환
    columns=Xtrain_ohe.columns,
    index=Xtrain_ohe.index
)
Xtest_ohe_scaled = pd.DataFrame(
    data=minmax_scaler.transform(Xtest_ohe),        # test: 변환만!
    columns=Xtest_ohe.columns,
    index=Xtest_ohe.index
)
print('스케일링 완료! 크기:', Xtrain_ohe_scaled.shape)

### (8) Target 로그 변환 — 분포가 치우쳐 있을 때!

```
대부분 값이 400~1000초인데 극소수가 80000초처럼 튀는 경우,
AI가 그 튀는 값에 과도하게 집중해버립니다.

로그 변환 = 큰 숫자 차이를 줄여줌
  1000 → 6.9,  10000 → 9.2,  100000 → 11.5  (차이가 훨씬 작아짐!)

Step2에서 오른쪽 그래프가 더 종 모양이면 → 로그 변환 사용!
평가지표가 RMSLE이면 → 무조건 로그 변환!
```

In [ ]:
# log1p = log(값+1) 변환  (0이 있어도 처리 가능)
ytrain_log = np.log1p(ytrain)

print('변환 전 최댓값:', ytrain.max())
print('변환 후 최댓값:', ytrain_log.max(), '  (훨씬 작아짐!)')

# ★ 로그 변환 안 할 때는: ytrain_log = ytrain  (그냥 같은 값 사용)

---
## [Step4] AI 모델 학습 + 평가

### (1) 회귀 평가지표 함수 만들기

```
회귀 평가지표는 종류가 여러 가지입니다.
문제에서 요구하는 평가지표를 사용하세요.

RMSLE: 평가지표가 RMSLE이면 이것 사용
RMSE:  평가지표가 RMSE이면 이것 사용
```

In [ ]:
# ★ RMSLE (Root Mean Squared Log Error)
# 로그 변환한 실제값과 예측값의 차이를 구하는 함수
def rmsle(log1p_actual, log1p_pred):
    squared_error = (log1p_actual - log1p_pred) ** 2
    return np.sqrt(np.mean(squared_error))

# ★ RMSE (Root Mean Squared Error)
def rmse(actual, pred):
    return np.sqrt(mean_squared_error(actual, pred))

# MAE, R²는 sklearn 함수 그대로 사용:
# mean_absolute_error(y_val, pred)
# r2_score(y_val, pred)

### (2) train / val 분리

In [ ]:
# ★ 로그 변환했으면 ytrain_log 사용, 안 했으면 ytrain 사용
X_train, X_val, y_train, y_val = train_test_split(
    Xtrain_ohe_scaled, ytrain_log,   # ★ 로그 변환 안 했으면 ytrain_log → ytrain
    test_size=0.2,
    random_state=0
)
print('학습용:', X_train.shape, '  검증용:', X_val.shape)

### (3) 여러 모델 학습 → 가장 좋은 모델 찾기

```
★ 회귀: 평가지표(RMSLE, RMSE 등)가 낮을수록 좋음!
         0에 가까울수록 예측이 정확

실행 후 가장 마지막에 출력된 '최종 선택 모델'이 최선입니다.
그 모델의 파라미터가 자동으로 출력됩니다 → GridSearchCV에 복붙!
```

In [ ]:
models = {
    'LinearRegression':      LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'RandomForestRegressor': RandomForestRegressor(n_jobs=-1),
    'ExtraTreesRegressor':   ExtraTreesRegressor(n_jobs=-1),
    'XGBRegressor':          XGBRegressor(verbose=-1, n_jobs=-1),
    'LGBMRegressor':         LGBMRegressor(verbose=-1, n_jobs=-1)
}

selected_estimator = None
min_score = float('inf')  # 회귀: 낮을수록 좋으니 무한대로 시작

for name, estimator in models.items():
    estimator.fit(X_train, y_train)       # 학습
    pred  = estimator.predict(X_val)      # 예측
    score = rmsle(y_val, pred)            # ★ 평가지표 함수 (문제에 맞게 변경)
    print(f'[{name}] rmsle: {round(score, 4)}')

    if score < min_score:                 # 더 낮으면 교체
        selected_estimator = estimator
        min_score = score
        print(f'  ★ 현재 최선!')
    print()

print('=' * 50)
print('최종 선택 모델:', type(selected_estimator).__name__)

# 선택된 모델의 권장 파라미터 자동 출력
PARAM_GUIDE = {
    'LGBMRegressor':         {'learning_rate': [0.05, 0.1, 0.14], 'n_estimators': [100, 200, 300]},
    'XGBRegressor':          {'learning_rate': [0.05, 0.1, 0.2],  'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7]},
    'RandomForestRegressor': {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'ExtraTreesRegressor':   {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'DecisionTreeRegressor': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
}
model_name = type(selected_estimator).__name__
if model_name in PARAM_GUIDE:
    print(f'\n[{model_name}] 아래 파라미터를 GridSearchCV에 복붙하세요:')
    print('parameters = {')
    for k, v in PARAM_GUIDE[model_name].items():
        print(f"    '{k}': {v},")
    print('}')

### (4) 하이퍼파라미터 튜닝 (GridSearchCV)

```
위에서 출력된 파라미터를 아래 parameters 딕셔너리에 복붙하세요!

★ 회귀: make_scorer 사용 (분류와 다른 점!)
   greater_is_better=False → 낮을수록 좋은 지표임을 알려줌
```

In [ ]:
# 먼저 기본 파라미터로 CV 성능 확인
grid_reg = GridSearchCV(
    selected_estimator, param_grid={}, cv=5,
    scoring=make_scorer(rmsle, greater_is_better=False)
)
grid_reg.fit(Xtrain_ohe_scaled, ytrain_log)
print('기본 파라미터 CV 성능 (RMSLE):', abs(round(grid_reg.best_score_, 4)))

In [ ]:
# ★ 위 모델 for문 결과에서 출력된 파라미터를 여기에 복붙!
parameters = {
    'learning_rate': [0.05, 0.1, 0.14],
    'n_estimators':  [100, 200, 300]
}

grid_reg = GridSearchCV(
    selected_estimator,
    param_grid=parameters,
    cv=5,
    scoring=make_scorer(rmsle, greater_is_better=False)  # ★ 회귀
)
grid_reg.fit(Xtrain_ohe_scaled, ytrain_log)

print('최적 파라미터:', grid_reg.best_params_)
print('최적 CV 성능 (RMSLE):', abs(round(grid_reg.best_score_, 4)))

---
## [Step5] 테스트 데이터 예측

In [ ]:
# 튜닝된 최선 모델로 test 예측
pred_test = grid_reg.best_estimator_.predict(Xtest_ohe_scaled)

# 튜닝 생략한 경우: pred_test = selected_estimator.predict(Xtest_ohe_scaled)

print('예측값 샘플 (로그 상태):', pred_test[:5])

In [ ]:
# ★ 로그 변환 했다면 → 반드시 역변환! (안 하면 제출값이 이상함)
# expm1 = exp(값) - 1  (log1p의 반대)
pred_test = np.expm1(pred_test)

# ★ 로그 변환 안 했다면 이 셀 건너뛰기!

print('역변환 후 예측값 샘플 (원래 단위):', pred_test[:5])

In [ ]:
# train 실제값 vs test 예측값 분포 비교 (비슷해야 정상!)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=ytrain,    ax=axes[0]).set_title('Train 실제값 분포')
sns.histplot(x=pred_test, ax=axes[1]).set_title('Test 예측값 분포')
plt.tight_layout()
plt.show()

---
## [Step6] 제출 파일 만들기

In [ ]:
submission = pd.read_csv('./data/sample_submission.csv')
submission.head()

In [ ]:
# ★ 제출 파일의 target 컬럼명으로 변경
TARGET_COL = 'trip_duration'
submission[TARGET_COL] = pred_test
submission.head(10)

In [ ]:
submission.to_csv('./data/MySubmission.csv', index=False)
print('✅ 제출 파일 저장 완료! → data/MySubmission.csv')